# 黑白图像骨架提取实验（学生练习版）

本 Notebook 是学生练习版：实验图像、基础形态学函数、可视化和结果对比代码已经保留；两种骨架提取算法的关键实现被改为 `TODO`。

本实验实现二值图像中的骨架提取。骨架可以看作目标区域的“中心线”或“拓扑结构”，能够在尽量保留目标形状连通关系的同时，减少前景像素数量。

本实验实现两种骨架提取算法：

1. **基于连续腐蚀和开运算差分的形态学骨架提取算法**
2. **Zhang-Suen 细化算法**

实验将先介绍两个算法的相关知识，再使用 Python 从零实现，并对同一幅黑白图像进行骨架提取对比。

## 1. 相关背景知识

### 1.1 什么是骨架

对于一个二值前景目标，骨架是目标区域内部的一组细线结构。它通常具有以下特点：

- 位于目标的中心附近。
- 尽量保持原目标的拓扑结构。
- 前景像素数量远少于原图。
- 可以用于形状识别、字符识别、道路中心线提取、医学图像结构分析等任务。

### 1.2 算法一：连续腐蚀与开运算差分

该算法基于数学形态学。核心思想是：

1. 对原图进行连续腐蚀，使目标一层一层向内收缩。
2. 每一层腐蚀图像记为 `E_k`。
3. 对 `E_k` 做开运算，得到 `Opening(E_k)`。
4. 用 `E_k - Opening(E_k)` 得到当前层中不能被开运算恢复的中心结构像素。
5. 将所有层的骨架像素取并集，得到最终骨架。

可写为：

`S(A) = union_k [E_k - Opening(E_k)]`

其中：

- `A` 是原始二值图像。
- `E_k` 是第 `k` 次腐蚀后的图像。
- `Opening(E_k)` 表示对 `E_k` 先腐蚀再膨胀。

该方法的优点是和形态学理论联系紧密，缺点是骨架可能较粗，且结果受结构元素影响明显。

### 1.3 算法二：Zhang-Suen 细化算法

Zhang-Suen 算法是一种经典二值图像细化算法。它通过迭代删除目标边界上的某些像素，使前景逐渐变细，最后得到单像素宽度的骨架。

每轮迭代分为两个子步骤：

1. 第一步扫描所有前景像素，找出满足删除条件的像素并删除。
2. 第二步再次扫描，使用另一组条件删除另一方向上的边界像素。

删除某个像素时，需要保证：

- 该像素不是孤立点。
- 删除它不会破坏目标连通性。
- 删除过程从边界向内部推进。

Zhang-Suen 算法通常可以得到较细的单像素骨架，适合字符、线状目标和轮廓较清晰的二值图像。

## 2. 实验图像设计

本实验直接生成一幅黑白二值图像，图像中包含：

- 粗线条路径
- 矩形区域
- 圆形区域
- T 形和 L 形结构

这样可以观察两种骨架算法在不同形状上的效果。

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

plt.rcParams["font.sans-serif"] = ["Microsoft YaHei", "SimHei", "Arial Unicode MS", "DejaVu Sans"]
plt.rcParams["axes.unicode_minus"] = False


def show_binary_image(image, title="", ax=None):
    """显示二值图像。"""
    if ax is None:
        _, ax = plt.subplots(figsize=(5, 5))
    ax.imshow(image, cmap="gray", vmin=0, vmax=1)
    ax.set_title(title)
    ax.axis("off")


def show_image_grid(images, titles, main_title="", cols=3, figsize=(12, 6)):
    """以网格方式显示多张二值图像。"""
    rows = int(np.ceil(len(images) / cols))
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).reshape(-1)
    for ax, image, title in zip(axes, images, titles):
        show_binary_image(image, title, ax)
    for ax in axes[len(images):]:
        ax.axis("off")
    fig.suptitle(main_title, fontsize=16)
    plt.tight_layout()
    plt.show()


def create_skeleton_test_image(size=128):
    """生成骨架提取实验图像。"""
    image = np.zeros((size, size), dtype=np.uint8)
    yy, xx = np.mgrid[0:size, 0:size]

    # 粗水平和垂直路径
    image[20:34, 12:88] = 1
    image[20:82, 42:56] = 1

    # L 形粗结构
    image[76:104, 18:32] = 1
    image[92:106, 18:76] = 1

    # T 形粗结构
    image[54:68, 76:116] = 1
    image[54:108, 94:108] = 1

    # 圆形目标
    circle = (xx - 96) ** 2 + (yy - 28) ** 2 <= 18 ** 2
    image[circle] = 1

    # 矩形目标
    image[82:114, 78:112] = 1
    return image


binary_image = create_skeleton_test_image()
show_binary_image(binary_image, "骨架提取实验二值图像")
plt.show()

## 3. 基础形态学函数

连续腐蚀和开运算差分算法需要用到：

- 腐蚀
- 膨胀
- 开运算

下面使用方形结构元素实现这些基础操作。

In [ ]:
def create_square_structuring_element(size=3):
    """创建方形结构元素。"""
    if size % 2 == 0:
        raise ValueError("结构元素大小必须是奇数")
    return np.ones((size, size), dtype=bool)


def binary_dilation(image, se):
    """二值膨胀。"""
    pad_y, pad_x = se.shape[0] // 2, se.shape[1] // 2
    padded = np.pad(image, ((pad_y, pad_y), (pad_x, pad_x)), mode="constant", constant_values=0)
    output = np.zeros_like(image, dtype=np.uint8)
    for y in range(image.shape[0]):
        for x in range(image.shape[1]):
            region = padded[y:y + se.shape[0], x:x + se.shape[1]]
            output[y, x] = 1 if np.any(region[se] == 1) else 0
    return output


def binary_erosion(image, se):
    """二值腐蚀。"""
    pad_y, pad_x = se.shape[0] // 2, se.shape[1] // 2
    padded = np.pad(image, ((pad_y, pad_y), (pad_x, pad_x)), mode="constant", constant_values=0)
    output = np.zeros_like(image, dtype=np.uint8)
    for y in range(image.shape[0]):
        for x in range(image.shape[1]):
            region = padded[y:y + se.shape[0], x:x + se.shape[1]]
            output[y, x] = 1 if np.all(region[se] == 1) else 0
    return output


def binary_opening(image, se):
    """二值开运算：先腐蚀，再膨胀。"""
    return binary_dilation(binary_erosion(image, se), se)

## 4. 算法一：连续腐蚀与开运算差分骨架

下面实现形态学骨架提取。

每一轮包含：

1. 当前图像 `current`。
2. 对 `current` 做开运算，得到 `opened`。
3. 计算 `layer = current - opened`，作为当前层骨架像素。
4. 将 `layer` 合并到总骨架中。
5. 对 `current` 继续腐蚀，进入下一层。

当腐蚀结果为空时停止。

### 4.1 算法一补全步骤

请按下面顺序完成连续腐蚀与开运算差分骨架算法：

1. 初始化 `current` 为原图副本，初始化空骨架 `skeleton`。
2. 当 `current` 仍有前景像素时，执行循环。
3. 对 `current` 做开运算，得到 `opened`。
4. 计算当前层骨架像素 `layer = current - opened`。
5. 将 `layer` 合并到 `skeleton`。
6. 对 `current` 做一次腐蚀，进入下一层。
7. 当腐蚀结果为空时停止，返回最终骨架和每层骨架像素。

In [ ]:
def morphology_skeleton(image, se, max_iterations=100):
    """通过连续腐蚀和开运算差分提取骨架。"""
    # TODO 1：复制输入图像作为 current。
    # TODO 2：创建与 image 同大小的空 skeleton，并创建 layers 列表。
    # TODO 3：循环执行，直到 current 中没有前景像素，或达到 max_iterations。
    # TODO 4：对 current 做开运算，得到 opened。
    # TODO 5：计算 layer = current - opened，并限制结果为 0/1。
    # TODO 6：将 layer 合并到 skeleton 中。
    # TODO 7：把 layer 加入 layers。
    # TODO 8：对 current 做腐蚀，进入下一轮。
    # TODO 9：返回 skeleton 和 layers。
    raise NotImplementedError("请补全 morphology_skeleton 函数")


se = create_square_structuring_element(3)
morph_skeleton, morph_layers = morphology_skeleton(binary_image, se)

print(f"形态学骨架层数：{len(morph_layers)}")
print(f"原图前景像素数：{int(binary_image.sum())}")
print(f"形态学骨架像素数：{int(morph_skeleton.sum())}")

show_image_grid(
    [binary_image, morph_skeleton],
    ["原始二值图像", "连续腐蚀 + 开运算差分骨架"],
    main_title="算法一：形态学骨架提取结果",
    cols=2,
    figsize=(9, 4),
)

In [ ]:
def show_skeleton_layers(layers, max_layers=8):
    """显示前若干层骨架像素。"""
    show_layers = layers[:max_layers]
    titles = [f"第 {i + 1} 层骨架像素" for i in range(len(show_layers))]
    show_image_grid(
        show_layers,
        titles,
        main_title="连续腐蚀过程中每一层提取到的骨架像素",
        cols=4,
        figsize=(12, 6),
    )


show_skeleton_layers(morph_layers, max_layers=8)

## 5. 算法二：Zhang-Suen 细化算法

Zhang-Suen 算法会反复删除边界点。对于某个前景像素 `P1`，按照顺时针方向定义 8 个邻居：

```text
P9 P2 P3
P8 P1 P4
P7 P6 P5
```

每个子迭代会检查：

1. `P1` 的前景邻居数量是否在合理范围内。
2. 从 `P2` 到 `P9` 再回到 `P2` 的 0 到 1 转换次数是否等于 1。
3. 若干方向约束是否满足，以避免破坏连通性。

满足条件的边界像素会被删除。算法不断迭代，直到没有像素可以删除。

### 5.1 算法二补全步骤

请按下面顺序完成 Zhang-Suen 细化算法：

1. 阅读 `get_neighbors(image, y, x)`，明确 P2 到 P9 的邻域顺序。
2. 使用 `count_foreground_neighbors` 统计 8 邻域前景数量。
3. 使用 `count_zero_to_one_transitions` 统计环形邻域中的 0 到 1 转换次数。
4. 在子迭代 1 中，按 Zhang-Suen 第一组条件找出待删除像素。
5. 删除子迭代 1 中找到的像素。
6. 在子迭代 2 中，按 Zhang-Suen 第二组条件找出待删除像素。
7. 删除子迭代 2 中找到的像素。
8. 如果某一轮两个子迭代都没有删除像素，则算法收敛并停止。

In [ ]:
def count_foreground_neighbors(neighbors):
    """计算 8 邻域中的前景数量。"""
    return sum(neighbors)


def count_zero_to_one_transitions(neighbors):
    """计算环形邻域序列中 0 到 1 的转换次数。"""
    sequence = neighbors + [neighbors[0]]
    transitions = 0
    for i in range(len(neighbors)):
        if sequence[i] == 0 and sequence[i + 1] == 1:
            transitions += 1
    return transitions


def get_neighbors(image, y, x):
    """按照 Zhang-Suen 定义返回 P2 到 P9。"""
    p2 = image[y - 1, x]
    p3 = image[y - 1, x + 1]
    p4 = image[y, x + 1]
    p5 = image[y + 1, x + 1]
    p6 = image[y + 1, x]
    p7 = image[y + 1, x - 1]
    p8 = image[y, x - 1]
    p9 = image[y - 1, x - 1]
    return [p2, p3, p4, p5, p6, p7, p8, p9]


def zhang_suen_thinning(image, max_iterations=200):
    """使用 Zhang-Suen 算法提取单像素骨架。"""
    # TODO 1：复制输入图像作为 thinned。
    # TODO 2：循环执行最多 max_iterations 轮。
    # TODO 3：子迭代 1：遍历非边界前景像素，计算邻居数量 n 和转换次数 s。
    # TODO 4：根据第一组条件选择要删除的像素：
    #         2 <= n <= 6, s == 1, p2*p4*p6 == 0, p4*p6*p8 == 0。
    # TODO 5：删除子迭代 1 选中的像素。
    # TODO 6：子迭代 2：重新遍历并使用第二组条件：
    #         2 <= n <= 6, s == 1, p2*p4*p8 == 0, p2*p6*p8 == 0。
    # TODO 7：删除子迭代 2 选中的像素。
    # TODO 8：如果一轮中没有任何像素被删除，则返回 thinned 和迭代轮数。
    raise NotImplementedError("请补全 zhang_suen_thinning 函数")


zs_skeleton, zs_iterations = zhang_suen_thinning(binary_image)

print(f"Zhang-Suen 迭代轮数：{zs_iterations}")
print(f"Zhang-Suen 骨架像素数：{int(zs_skeleton.sum())}")

show_image_grid(
    [binary_image, zs_skeleton],
    ["原始二值图像", "Zhang-Suen 细化骨架"],
    main_title="算法二：Zhang-Suen 骨架提取结果",
    cols=2,
    figsize=(9, 4),
)

## 6. 两种算法结果对比

下面将两种算法结果放在一起观察。

通常情况下：

- 形态学骨架算法更能体现“逐层腐蚀”的数学形态学思想。
- Zhang-Suen 算法得到的骨架通常更细，更接近单像素中心线。
- 两者的骨架像素数量、连通性和视觉效果可能不同。

In [ ]:
def overlay_skeleton(image, skeleton):
    """将骨架用红色叠加到原图上。"""
    rgb = np.dstack([image, image, image]).astype(float)
    rgb[skeleton == 1] = [1.0, 0.0, 0.0]
    return rgb


fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
show_binary_image(binary_image, "原始图像", axes[0])
axes[1].imshow(overlay_skeleton(binary_image, morph_skeleton))
axes[1].set_title("形态学骨架叠加显示")
axes[1].axis("off")
axes[2].imshow(overlay_skeleton(binary_image, zs_skeleton))
axes[2].set_title("Zhang-Suen 骨架叠加显示")
axes[2].axis("off")
plt.tight_layout()
plt.show()

print("对比统计：")
print(f"原图前景像素数：{int(binary_image.sum())}")
print(f"形态学骨架像素数：{int(morph_skeleton.sum())}")
print(f"Zhang-Suen 骨架像素数：{int(zs_skeleton.sum())}")

## 7. 实验思考

完成实验后，可以思考下面的问题：

1. 连续腐蚀与开运算差分算法中，为什么要计算 `E_k - Opening(E_k)`？
2. 为什么要把所有层的骨架像素联合起来？
3. 结构元素大小会如何影响形态学骨架结果？
4. Zhang-Suen 算法为什么需要两个子迭代？
5. Zhang-Suen 算法删除边界像素时，为什么要保护连通性？
6. 两种算法得到的骨架有什么差异？哪一种更接近单像素中心线？